# SEC API in Python: requests, paging, retries and pandas

A small Python client for SEC filings, Form 4 trades, 13F holdings and financials: one search function, paging past 50 results, backing off on 429, and a DataFrame at the end.

Read it with charts at https://www.quantisnow.com/sec-api/guides/sec-api-python. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Every endpoint takes the same body, `query`, `from`, `size` and `sort`, and returns `total` and `data`. So one function covers all of them. All you need is `requests` and an [API key](https://www.quantisnow.com/sec-api/keys).

```shell
pip install requests pandas
export QUANTISNOW_API_KEY=qs_your_key
```

## A client in 30 lines

In [ ]:
import os, time, requests

BASE = "https://www.quantisnow.com/api/v1"
session = requests.Session()
session.headers["Authorization"] = f"Bearer {os.environ['QUANTISNOW_API_KEY']}"

def search(endpoint, query, size=50, start=0, sort=None):
    body = {"query": query, "size": size, "from": start}
    if sort:
        body["sort"] = sort
    for attempt in range(5):
        res = session.post(f"{BASE}/{endpoint}", json=body, timeout=30)
        if res.status_code == 429 and res.json()["error"]["code"] == "rate_limited":
            time.sleep(int(res.headers.get("retry-after", "1")))
            continue
        if not res.ok:
            raise RuntimeError(res.json()["error"]["message"])
        return res.json()
    raise RuntimeError("still rate limited after 5 tries")

def search_all(endpoint, query, sort=None, limit=10_000):
    rows, start = [], 0
    while start < limit:
        page = search(endpoint, query, start=start, sort=sort)
        rows += page["data"]
        start += 50
        if start >= page["total"]["value"]:
            break
    return rows

Two kinds of 429 come back. `rate_limited` means too many requests in one second and is worth a retry after `retry-after`. `quota_exceeded` means the month's calls and your credits are used up, and retrying won't help.

## Use it

In [ ]:
import pandas as pd

filings = search_all("filings", "ticker:NVDA AND filedAt:[2026-01-01 TO 2026-11-23]", sort="filedAt:desc")
df = pd.DataFrame(filings)
print(df.groupby("formType").size().sort_values(ascending=False))

That query returns records like these:

### What has Nvidia filed with the SEC in the last 90 days?

A company's whole filing history for a quarter, newest first, to diff against what you already have.

```shell
curl -X POST https://www.quantisnow.com/api/v1/filings \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "ticker:NVDA AND filedAt:[2026-08-26 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"filedAt": {"order": "desc"}}]
}'
```

Response (real, first 2 of 25):

```json
{
  "total": {
    "value": 25,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001696841-26-000016",
      "accessionNo": "0001696841-26-000016",
      "formType": "4",
      "filedAt": "2026-11-23T16:54:28-05:00",
      "cik": "0001045810",
      "companyName": "NVIDIA Corporation",
      "ticker": "NVDA",
      "tickers": [
        "NVDA"
      ],
      "items": [],
      "description": "EVP, General Counsel and Sec Timothy S. Teter sold $4,542,781 worth of shares (30,000 units at $151.43) as part of a pre-agreed trading plan (SEC Form 4)",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1045810/000169684126000016/0001696841-26-000016-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/evp-general-counsel-and-sec-timothy-s-teter-sold-4542781-6804297"
    },
    {
      "id": "0001045810-26-000090",
      "accessionNo": "0001045810-26-000090",
      "formType": "8-K",
      "filedAt": "2026-11-23T08:57:54-05:00",
      "cik": "0001045810",
      "companyName": "NVIDIA Corporation",
      "ticker": "NVDA",
      "tickers": [
        "NVDA"
      ],
      "items": [
        "5.02",
        "9.01"
      ],
      "description": "NVIDIA Corporation filed SEC Form 8-K: Leadership Update, Financial Statements and Exhibits",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1045810/000104581026000090/0001045810-26-000090-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/nvidia-corporation-filed-sec-form-8-k-leadership-update-financial-statements-6803386"
    }
  ]
}
```

## Know what you have left

Every response carries `x-ratelimit-remaining`, the plan calls left this month. When a call is paid from credits (the free calls are credits), `x-credits-remaining` shows the balance. `GET /v1/usage` returns both and is free.

In [ ]:
res = session.post(f"{BASE}/filings", json={"query": "formType:10-K", "size": 1})
print(res.headers["x-ratelimit-remaining"], res.headers.get("x-credits-remaining"))

Searching inside the text of filings instead? See [keyword search in Python](https://www.quantisnow.com/sec-api/guides/search-sec-filings-python). More than 10,000 results? See [paging past 10,000](https://www.quantisnow.com/sec-api/guides/paginate-past-10000-results).